In [ ]:
#PACKAGES: - double check might not need all of these
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import time_support
from astropy.time import Time
import astropy.units as u

from sunpy import timeseries as ts
from sunpy.net import Fido
from sunpy.net import attrs as a

from stixpy.net.client import STIXClient
from stixpy.timeseries import quicklook 
from stixpy.product import Product

import datetime as dt
from sunpy.time import parse_time
from sunpy.time import TimeRange

import pandas as pd

from scipy.signal import find_peaks, savgol_filter

from matplotlib import dates

In [ ]:
#loading in flare list for reference/locating flares
flares = pd.read_csv("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/data/STIX_flarelist_w_locations_20210101_20260130_version1_python.csv")

In [ ]:
#so far have used top 100 flares by GOES estimated flux as starting point
top100 = flares.sort_values("goes_estimated_mean_flux", ascending=False).head(100)

In [ ]:
#USEFUL FUNCTIONS:
def t(string):
    return parse_time(string).datetime

def get_energy_indices(qtable, ranges):
    energy_indices = []
    for e_min, e_max in ranges:
        start_index = np.where(qtable["e_low"] >= e_min)[0][0]
        end_index = np.where(qtable["e_high"] <= e_max)[0][-1]
        energy_indices.append([int(start_index), int(end_index)])
    return energy_indices
    
def get_stix_df(stix_sci, energy_ranges):
    
    energy_indices = get_energy_indices(stix_sci.energies, energy_ranges)
    counts, errors, times, timedeltas, energies = stix_sci.get_data(detector_indices=[[0, 31]],
                                                                    pixel_indices=[[0, 11]],
                                                                    energy_indices=energy_indices,)
    counts = counts.to(u.ct / u.s / u.keV)
    errors = errors.to(u.ct / u.s / u.keV)
    timedeltas = timedeltas.to(u.s)
    
    times = times
    
    energy_columns = [f"{e['e_low']}-{e['e_high']}" for e in energies]
    
    counts_reshaped = counts[:, 0, 0, :]
    
    counts_df = pd.DataFrame(counts_reshaped, index=times.datetime, columns=energy_columns)
    return counts_df

#might not need, check
def rank_overlaps(start, end, sci_query):
    overlaps=[]
    for n in range(len(sci_query[0])):
        latest_start = max(start, sci_query[0][n][0].datetime)
        earliest_end = min(end, sci_query[0][n][1].datetime)
        
        overlap = earliest_end - latest_start
        overlaps.append((overlap.total_seconds(), n))
        
    return np.array(sorted(overlaps, key=lambda x: x[0], reverse=True))

In [ ]:
#input list of desired flare list locations (index of flare list df - maybe change to take different argument?) and get list of dataframes from stix science spectrograph data 
#(could change to not spectrograph)
def get_spec_dfs(locs):
    df_list=[]
    for ind in locs:
        sci_query = Fido.search(a.Time(flares['start_UTC'].loc[ind], 
                                       flares['end_UTC'].loc[ind]), 
                            a.Instrument.stix,
                            a.stix.DataType.sci,
                            a.stix.DataProduct.sci_xray_spec)
        sci_query['stix'].filter_for_latest_version()
    
        start = parse_time(flares['start_UTC'].loc[ind]).datetime
        end = parse_time(flares['end_UTC'].loc[ind]).datetime
    
        overlaps_ranked = rank_overlaps(start, end, sci_query)
        
        for index in overlaps_ranked[:,1]:
            badrange=False
            sci_files = Fido.fetch(sci_query[0][int(index)])
            sci_data = Product(sci_files)
        
            if sci_data.energies["e_high"][len(sci_data.energies["e_high"])-1]<100*u.keV or sci_data.energies["e_low"][0]>25*u.keV:
                badrange=True
            
            if not badrange:
                sci_df = get_stix_df(sci_data, energy_ranges)
                df_list.append(sci_df)
                indices.append(ind)
                break
    return df_list

In [ ]:
#concactenates files for flares not covered by one
#currently requires handling manually - could incorporate into main function (is there every a case where need >2 files?)
for ind in incomplete_file_ind:
    sci_query = Fido.search(a.Time(top100['start_UTC'].iloc[ind], top100['end_UTC'].iloc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()
    sci_query

    sci_files = Fido.fetch(sci_query)
    sci_files = sorted(sci_files)
    
    
    sci_data_A = Product(sci_files[0])
    sci_data_B = Product(sci_files[1])
    
    df_A = get_stix_df(sci_data_A, energy_ranges)
    df_B = get_stix_df(sci_data_B, energy_ranges)
    
    combined_df = pd.concat([df_A, df_B])
    
    combined_df = combined_df.sort_index()
    
    df = combined_df[~combined_df.index.duplicated(keep="first")]

    df_list[ind]=df